In [ ]:
%load_ext autoreload
%autoreload 2

from itertools import product
import sys
sys.path.append('/home/projects/nyosef/zvise/PixelGen/')
from PixelGen.multimodalvi import MultiModalSCVI
from PixelGen.multimodalvae import MultiModalVAE, AggMethod, D
from PixelGen.enums import AggMethod, D
from PixelGen.metrics import MultiModalVIMetrics
from sklearn.preprocessing import PowerTransformer
from pathlib import Path


import anndata as ad
import pixelator
import torch
import scvi
import scipy
# from scvi import autotune

import seaborn as sns
import scanpy as sc
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection

from PixelGen.pxl_utils import train_model, get_model_latents
from PixelGen.scvi_utils import plot_losses, pca_neighbors_umap, calc_PCA
from pixelator.common.statistics import clr_transformation, dsb_normalize

from pixelator.pna.plot import molecule_rank_plot

print(torch.cuda.is_available())
from sklearn.decomposition import PCA

from anndata import AnnData
scvi.settings.seed = 0
print("Last run with scvi-tools version:", scvi.__version__)
sc.set_figure_params(figsize=(6, 6), frameon=False)

sns.set_theme()
torch.set_float32_matmul_precision("high")

from utils import plot_latent, plot_gene_heatmap, plot_model_latents
from doublet_seperation import B_CD4_logfc_dict, B_CD8_logfc_dict, run_cellwise_coloc_analysis_to_disk, concat_abundance_to_adata, add_doublets_metadata
%config InlineBackend.print_figure_kwargs={"facecolor": "w"}
%config InlineBackend.figure_format="retina"
from pixelator import read_pna as read
import pickle
import scipy.sparse as sp


import cellcharter as cc
import plotly.express as px



In [ ]:
import glob
import random
import os
path='/home/projects/nyosef/zvise/PixelGen/PixelGen/cache/GVAE/graphs'
files = glob.glob(f"{path}/*.pt")
if not files:
    print("❌ No files found in 'processed_cells/'. Check your saving loop.")
else:
    print(f"✅ Found {len(files)} files.")
    
    # 2. Inspect 3 random files
    print("\n--- Inspecting 3 Random Samples ---")
    for i in range(3):
        f = random.choice(files)
        data = torch.load(f)
        
        print(f"\n📂 File: {os.path.basename(f)}")
        print(f"   • Nodes: {data.num_nodes}")
        print(f"   • Edges: {data.num_edges}")
        print(f"   • Feature Matrix (x): {data.x.shape}  <-- Should be [Nodes, 159]")
        print(f"   • Connectivity (edge_index): {data.edge_index.shape} <-- Should be [2, Edges]")
        
        # 3. Sanity Checks
        # Check if features are actually One-Hot (Sum should be 1.0 per node)
        is_one_hot = torch.allclose(data.x.sum(dim=1), torch.tensor(1.0))
        print(f"   • Is One-Hot Valid? {is_one_hot}")
        
        # Check max edge index matches num_nodes
        if data.num_edges > 0:
            max_idx = data.edge_index.max().item()
            print(f"   • Max Node ID in edges: {max_idx} (Should be < {data.num_nodes})")

        # Peek at the first node's feature
        # finding which protein is active for node 0
        active_protein_idx = data.x[0].argmax().item()
        print(f"   • Node 0 is Protein Index: {active_protein_idx}")

In [ ]:
ANNOTATED_ADATA_PATH='/home/projects/nyosef/zvise/PixelGen/PixelGen/Data/adatas/final_adatas/adata_annotated.h5ad'
DATA_DIR = Path("/home/projects/nyosef/zvise/PxlgnProject/Data")
DOUBLETS_SEP_CACHE='/home/projects/nyosef/zvise/PixelGen/PixelGen/cache/doublet_separation_run'
SEPARATION_PKL_PATH= '/home/projects/nyosef/zvise/PixelGen/PixelGen/cache/results.pkl'
T_CELLS_ADATA_PATH='/home/projects/nyosef/zvise/PixelGen/PixelGen/Data/adatas/final_adatas/t_cell_adata.h5ad'


files = [f for f in DATA_DIR.rglob('*.pxl') if f.is_file()]
pg_data = read(files)
adata=sc.read_h5ad(ANNOTATED_ADATA_PATH)

In [ ]:
cell=adata.obs_names[99]
edge_list=pg_data.filter(components=cell).edgelist().to_df()
layout=pg_data.filter(components=cell).precomputed_layouts().to_df()
edge_list.head()

In [ ]:
import networkx as nx
G = nx.from_pandas_edgelist(
    edge_list,
    source="umi1",
    target="umi2"
)
nx.is_connected(G)


In [ ]:
import pandas as pd
import numpy as np
import anndata as ad
from scipy import sparse
from sklearn.preprocessing import OneHotEncoder

# 1. Load your data (assuming it's in a DataFrame named 'df')
# If loading from a file: df = pd.read_csv("your_file.csv", sep="\t")
# For this example, I'll assume 'df' contains the columns from your snippet.
df=edge_list.copy()
# 2. Extract unique Nodes (UMIs) and their Markers
# We stack umi1/marker_1 and umi2/marker_2 to get a master list of all molecules
node_list_1 = df[['umi1', 'marker_1', 'component', 'sample']].rename(
    columns={'umi1': 'umi', 'marker_1': 'marker'}
)
node_list_2 = df[['umi2', 'marker_2', 'component', 'sample']].rename(
    columns={'umi2': 'umi', 'marker_2': 'marker'}
)

# Combine and drop duplicates (a UMI might appear in multiple edges)
nodes = pd.concat([node_list_1, node_list_2]).drop_duplicates(subset='umi').reset_index(drop=True)

# 3. Create the Feature Matrix (X)
# CellCharter needs features. For molecules, the "feature" is the protein identity.
# We one-hot encode the marker names (e.g., Is_CD44, Is_B2M).
encoder = OneHotEncoder(sparse_output=False)
X_data = encoder.fit_transform(nodes[['marker']])
var_names = encoder.get_feature_names_out(['marker'])

# 4. Construct the Adjacency Matrix (The Graph)
# Map UMIs to their new DataFrame indices
umi_to_idx = {umi: i for i, umi in enumerate(nodes['umi'])}

# Map the edge list to these indices
row_indices = df['umi1'].map(umi_to_idx).values
col_indices = df['umi2'].map(umi_to_idx).values

# Create symmetric adjacency matrix (if UMI A touches B, B touches A)
# Combine (row, col) and (col, row) to ensure symmetry
rows = np.concatenate([row_indices, col_indices])
cols = np.concatenate([col_indices, row_indices])
data = np.ones(len(rows))

adj_matrix = sparse.csr_matrix(
    (data, (rows, cols)), 
    shape=(len(nodes), len(nodes))
)
# Binarize to ensure weights are 1 (just connectivity)
adj_matrix.data[:] = 1

# 5. Create AnnData Object
adata = ad.AnnData(X=X_data, obs=nodes, var=pd.DataFrame(index=var_names))
adata.obsp['spatial_connectivities'] = adj_matrix
adata.obsp['spatial_distances'] = adj_matrix  # Optional, if needed by some tools

# Set 'component' as the batch/sample key if you want to analyze domains per cell
# or 'sample' if you want to analyze across the whole experiment.
adata.obs['sample_key'] = adata.obs['component'].astype(str)

print(adata)

In [ ]:
import cellcharter as cc
import scanpy as sc

# 1. Preprocessing (Optional for one-hot data, but often good to reduce dim)
# Since X is already low-dim one-hot, you might skip PCA or use simple PCA.
# CellCharter typically aggregates 'X_scVI' or similar. 
# Here we can just use the raw one-hot features (adata.X) or run PCA.
sc.pp.pca(adata, n_comps=min(10, adata.n_vars - 1))

# 2. Neighborhood Aggregation
# Aggregates the protein environment of each molecule (e.g., "I am a B2M molecule 
# and I am surrounded by CD44 and HLA-DR").
# We use the 'spatial_connectivities' we built manually.
cc.gr.aggregate_neighbors(
    adata, 
    n_layers=3,              # How many steps away to look (3 hops on the graph)
    use_rep='X_pca',         # Or 'X' if you didn't run PCA
    out_key='X_cellcharter',
    sample_key='sample_key'  # Use component/cell ID here to respect cell boundaries
)

# 3. Clustering (Identify Domains)
# This will group molecules that have similar neighborhoods.
model = cc.tl.ClusterAutoK(
    n_clusters=(2, 10),      # Search for 2 to 10 spatial domains
    max_runs=5
)
model.fit(adata, use_rep='X_cellcharter')

# Predict domains
adata.obs['molecular_niche'] = model.predict(adata, use_rep='X_cellcharter')

# 4. Visualization
# Since you don't have X,Y coordinates for every molecule (unless Pixelgen provided a layout),
# you can't use standard spatial scatter plots easily.
# However, you can analyze the composition of the niches:
sc.pl.dotplot(adata, var_names=adata.var_names, groupby='molecular_niche')

In [ ]:
sc.pp.neighbors(adata, use_rep='X_cellcharter', n_neighbors=15)

# 2. Run Leiden clustering with higher resolution
# Default is 1.0. Try 1.5 or 2.0 to break the 2 clusters apart.
sc.tl.leiden(adata, resolution=0.3, key_added='molecular_niche_leiden')

In [ ]:

results_df = adata.obs[['umi', 'molecular_niche','molecular_niche_leiden']].copy()

In [ ]:
results_df['umi'] = results_df['umi'].astype(str)

layout_merged = layout.copy()
layout_merged['index'] = layout_merged['index'].astype(str)

merged_df = pd.merge(
    layout_merged,
    results_df,
    left_on='index',   # The column in 'layout' with the UMI
    right_on='umi',    # The column in 'adata.obs' with the UMI
    how='left'         # Keep all layout points, even if they have no niche
)

merged_df

In [ ]:
# Create the 3D Scatter Plot
fig = px.scatter_3d(
    merged_df,
    x='x', 
    y='y', 
    z='z',
    color='molecular_niche_leiden',  # This comes from CellCharter
    
    # Optional: Add extra info on hover
    hover_data=['component'], 
    
    title='CellCharter Molecular Niches (3D Layout)',
    opacity=0.8,
    size_max=2,               # Keep dots small to see structure
    color_discrete_sequence=px.colors.qualitative.Bold # Distinct colors
)

# Refine visual style (remove background grid for cleaner look)
fig.update_layout(
    scene=dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False)
    )
)
fig.update_traces(marker=dict(size=1.5)) # Force small marker size
fig.write_html("/home/projects/nyosef/zvise/PixelGen/PixelGen/figures/molecular_niches_3d.html")

fig.show()

In [ ]:
fig = px.scatter_3d(
    merged_df,
    x='x', 
    y='y', 
    z='z',
    color='molecular_niche',
    title='CellCharter Molecular Niches',
    opacity=0.8,
    size_max=2,
    color_discrete_sequence=px.colors.qualitative.Bold
)

# Optional: Clean up the view (remove axes)
fig.update_layout(
    scene=dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False)
    )
)
fig.update_traces(marker=dict(size=1.5))

# 2. Save to HTML
fig.write_html("molecular_niches_3d.html")

print("Saved to molecular_niches_3d.html")